# 학습 영상 3장의 골반 수동 검수
b04 9번·21번과 a01 5번에서 **선수 본인의 왼쪽 골반**을 두 차례 짚습니다. 파일 번호는 영상 첫 장을 0번으로 센 값입니다. 2차는 순서를 바꿔 원본을 다시 보여 줍니다. 예비 판독 및 1차 좌표는 2차 화면에 표시되지 않습니다.

**모든 셀 실행(Run All Cells)** 후 맨 아래 화면을 사용하세요. 사진에서 고관절 소켓으로 추정한 위치를 한 번 누르면 저장됩니다. 옷에 가려 추정한 경우에는 먼저 **찍을 점 → 옷에 가려 추정**을 선택하세요. 위치를 정하기 어려우면 사진을 누르는 대신 **가려서 판독 불가** 또는 **위치 불확실**을 선택하세요. 한 장을 저장할 때마다 **다음 장면**을 누릅니다.

### 원본과 저장 위치 준비

In [ ]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from wlcv.annotation.pose_points import load_points, pose_point_editor, pose_point_path
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS

frames = [("b04_sn_f_01", 9), ("b04_sn_f_01", 21), ("a01_sn_f_01", 5)]
# 두 번째 회차는 순서를 한 칸 바꾼다. 중단 후 다시 열어도 같은 순서다.
orders = [frames, [frames[1], frames[2], frames[0]]]
folder = ANNOTATIONS / "train_hip_review"
videos = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id")
split = pd.read_csv(SPLIT).set_index("video_id")
for video, frame in frames:
    assert split.at[video, "role"] == "train" and videos.at[video, "near_side"] == "left"


### 골반 찍기
다시 실행하면 첫 미완료 장면에서 이어집니다. 좌표와 판독 불가 사유는 `data/annotations/train_hip_review/personal_pose_points/` 아래에 자동 저장되며, 2차 파일은 그 안의 `pass_2/`에 저장됩니다.

In [ ]:
panel = widgets.VBox()
progress = widgets.Label()
next_button = widgets.Button(description="다음 장면")

def show_next(_=None):
    completed = 0
    for round_number, order in enumerate(orders, start=1):
        for index, (video, frame) in enumerate(order, start=1):
            path = pose_point_path(folder, video, frame, pass_number=round_number)
            if "골반" in load_points(path):
                completed += 1
                continue
            info = videos.loc[video]
            label = f"{video} · {frame}번" if round_number == 1 else f"장면 {index}/3"
            editor = pose_point_editor(
                VIDEOS / info.filename, frame, info.person_id, info.near_side,
                split="train", annotations=folder, pass_number=round_number,
                joints=("골반",), trunk_review=True, frame_label=label,
            )
            panel.children = (editor,)
            progress.value = f"{completed}/6 저장됨 · 현재 장면을 판독한 뒤 다음 장면을 누르세요."
            return
    panel.children = ()
    progress.value = "6/6 저장됨 · 세 장의 1차·2차 수동 검수를 마쳤습니다."
    next_button.disabled = True

next_button.on_click(show_next)
show_next()
display(widgets.VBox([progress, panel, next_button]))